In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
import yaml
from torch.utils.data import DataLoader

In [2]:
def find_project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(6):
        if (p / "configs" / "config.yaml").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Could not find configs/config.yaml")


ROOT = find_project_root()
sys.path.insert(0, str(ROOT.parent))

In [3]:
from ebm_unlearning.src.data.dataset import DatasetSpec, load_dataset
from ebm_unlearning.src.data.split import ForgetSpec, RetainSpec, split_forget_retain, train_holdout_split
from ebm_unlearning.src.utils.seed import set_seed

In [4]:
print("Project root:", ROOT)

with open(ROOT / "configs" / "config.yaml", "r") as f:
    cfg = yaml.safe_load(f)
    cfg_path = ROOT / "configs" / "config.yaml"
    print("cfg_path:", cfg_path)
    print("cfg_file_first_lines:\n", cfg_path.read_text().splitlines()[:10])
    print("cfg_loaded_dataset:", cfg["data"]["dataset"])

print("Dataset:", cfg["data"]["dataset"], "Data dir:", cfg["data"]["data_dir"])

set_seed(int(cfg["seed"]))

device = torch.device(cfg.get("device", "cpu"))
print("Device:", device)

Project root: /home/owais/machine unlearning/ebm_unlearning
cfg_path: /home/owais/machine unlearning/ebm_unlearning/configs/config.yaml
cfg_file_first_lines:
 ['seed: 42', 'device: cuda', '', 'data:', '  dataset: cifar10  # mnist | cifar10', '  data_dir: ./data', '  batch_size: 128', '  num_workers: 2', '  forget:', '    mode: class']
cfg_loaded_dataset: cifar10
Dataset: cifar10 Data dir: ./data
Device: cuda


In [5]:
spec = DatasetSpec(
    name=cfg["data"]["dataset"],
    data_dir=str(ROOT / cfg["data"]["data_dir"]),
    train=True,
    download=True,
)
dset = load_dataset(spec)

forget_spec = ForgetSpec(mode=cfg["data"]["forget"]["mode"], class_label=cfg["data"]["forget"]["class_label"])
retain_spec = RetainSpec(mode=cfg["data"]["retain"]["mode"])

forget_all, retain_all = split_forget_retain(dset, forget_spec, retain_spec)

holdout_fraction = float(cfg["evaluation"]["holdout_fraction"])
forget_train, forget_holdout = train_holdout_split(forget_all, holdout_fraction, seed=int(cfg["seed"]))
retain_train, retain_holdout = train_holdout_split(retain_all, holdout_fraction, seed=int(cfg["seed"]) + 1)

print("forget_all:", len(forget_all))
print("retain_all:", len(retain_all))
print("forget_train/holdout:", len(forget_train), len(forget_holdout))
print("retain_train/holdout:", len(retain_train), len(retain_holdout))

[data] loading cifar10 (train=True, download=True) from /home/owais/machine unlearning/ebm_unlearning/data


100%|██████████| 170M/170M [01:57<00:00, 1.45MB/s] 


Extracting /home/owais/machine unlearning/ebm_unlearning/data/cifar-10-python.tar.gz to /home/owais/machine unlearning/ebm_unlearning/data
forget_all: 5000
retain_all: 45000
forget_train/holdout: 4000 1000
retain_train/holdout: 36000 9000


In [6]:
batch_size = int(cfg["data"]["batch_size"])
num_workers = int(cfg["data"]["num_workers"])

forget_train_loader = DataLoader(forget_train, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
retain_train_loader = DataLoader(retain_train, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)

xb, yb = next(iter(forget_train_loader))
print("Batch x shape:", tuple(xb.shape), "y shape:", tuple(yb.shape))
print("x range:", float(xb.min()), float(xb.max()))

Batch x shape: (128, 3, 32, 32) y shape: (128,)
x range: -1.9894737005233765 2.12648868560791
